# Strategic Asset Allocation (SAA) Derivation

This notebook builds the thesis universe, loads price histories, computes return and risk metrics, checks diversification in bull and bear regimes, and solves a long-only mean-variance model for the strategic asset allocation.


In [6]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / 'core').exists() and (project_root.parent / 'core').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from core.universe import build_michael_portfolio
from core.data_loader import get_price_history
from core.risk import var_cvar_summary
from analysis.correlation import bull_bear_correlation_summary, largest_correlation_increases

print('Imports loaded successfully.')
print(f'Project root: {project_root}')


Imports loaded successfully.
Project root: C:\Users\mginf\OneDrive\Documents\porfolio-saa-project


# 1) Build the universe and fetch price data

This section constructs the thesis portfolio, retrieves price histories, and confirms the asset universe and return matrix are aligned before any optimization is run.


In [ ]:
portfolio = build_michael_portfolio()
print(f"Portfolio loaded with {len(portfolio.securities)} securities.")

for security in portfolio.securities:
    security.fetch_prices(force_refresh=False)

print(f"Fetched prices for {sum(1 for s in portfolio.securities if s._prices is not None)} securities.")

available = [s for s in portfolio.securities if s._prices is not None]
close_matrix = pd.concat({s.ticker: s.close for s in available}, axis=1).dropna()
returns = close_matrix.pct_change().dropna()
benchmark_close = get_price_history('^GSPTSE')['Close'].dropna()
#print(close_matrix)

print('Price matrix shape:', close_matrix.shape)
print('Return matrix shape:', returns.shape)
print('Benchmark series head:')
print(benchmark_close.head())


Portfolio loaded with 11 securities.
Fetched prices for 11 securities.
Price matrix shape: (1163, 11)
Return matrix shape: (1162, 11)
Benchmark series head:
Date
1979-06-29    1618.400024
1979-07-03    1602.800049
1979-07-04    1591.400024
1979-07-05    1583.900024
1979-07-06    1586.599976
Name: Close, dtype: float64


# 2) Compute return and risk statistics

This section annualizes expected returns and volatility, calculates Sharpe ratios, and displays the side-by-side tail-risk summary for each asset before optimization.


In [5]:
annualized_return = returns.mean() * 252
annualized_vol = returns.std() * np.sqrt(252)
annualized_rf = 0.02
sharpe = (annualized_return - annualized_rf) / annualized_vol

risk_rows = []
for ticker in returns.columns:
    tail = var_cvar_summary(returns[ticker], confidence=0.95, n_boot=2000, seed=42)
    risk_rows.append({
        'ticker': ticker,
        'historical_VaR_95': float(tail.loc['historical', 'VaR']),
        'historical_CVaR_95': float(tail.loc['historical', 'CVaR']),
        'parametric_VaR_95': float(tail.loc['parametric', 'VaR']),
        'bootstrap_VaR_95': float(tail.loc['bootstrap', 'VaR']),
    })

asset_metrics = pd.DataFrame({
    'expected_return': annualized_return,
    'volatility': annualized_vol,
    'sharpe': sharpe,
}).join(pd.DataFrame(risk_rows).set_index('ticker')).sort_values('expected_return', ascending=False)

print('Annualized asset metrics:')
print(asset_metrics.head(10))


Annualized asset metrics:
           expected_return  volatility    sharpe  historical_VaR_95  \
CGL.TO            0.184605    0.192977  0.852979           0.018572   
XUU.TO            0.132127    0.159753  0.701881           0.015851   
XIC.TO            0.118407    0.136350  0.721726           0.014268   
AVUV              0.116936    0.228287  0.424623           0.022127   
VTV               0.101425    0.141680  0.574708           0.013620   
VIU.TO            0.100731    0.148535  0.543511           0.014325   
VEE.TO            0.067262    0.157279  0.300499           0.014970   
CASH.TO           0.000041    0.011196 -1.782698           0.000395   
XFR.TO           -0.000404    0.012821 -1.591462           0.000999   
CAR-UN.TO        -0.096911    0.216319 -0.540454           0.021663   

           historical_CVaR_95  parametric_VaR_95  bootstrap_VaR_95  
CGL.TO               0.028097           0.019263          0.018480  
XUU.TO               0.023078           0.016029      

# 3) Check diversification and regime dependence

This section compares full-period, bull-period, and bear-period correlations to see whether diversification remains valuable during stress.


In [ ]:
correlation_summary = bull_bear_correlation_summary(
    returns,
    benchmark_close,
    drawdown_threshold=0.10,
    min_bear_days_warning=60,
)

full_corr = correlation_summary['full'].round(3)
bull_corr = correlation_summary['bull'].round(3)
bear_corr = correlation_summary['bear'].round(3)
largest_rises = largest_correlation_increases(correlation_summary, top_n=10).round(3)

print('Full-period correlation matrix:')
print(full_corr)
print('\nBull-period correlation matrix:')
print(bull_corr)
print('\nBear-period correlation matrix:')
print(bear_corr)
print('\nLargest bear-vs-bull correlation rises:')
print(largest_rises)


# 4) Solve the long-only mean-variance optimization

This section estimates the efficient frontier under a long-only constraint and solves for the maximum-Sharpe and minimum-volatility allocations.


In [ ]:
mean_vec = annualized_return
cov_mat = returns.cov() * 252
weights0 = np.ones(len(mean_vec)) / len(mean_vec)
annualized_rf = 0.02


def portfolio_stats(w):
    w = np.asarray(w, dtype=float)
    exp_ret = float(w @ mean_vec)
    variance = float(w @ cov_mat @ w)
    volatility = float(np.sqrt(variance))
    sharpe = (exp_ret - annualized_rf) / volatility if volatility > 0 else 0.0
    return {'return': exp_ret, 'volatility': volatility, 'variance': variance, 'sharpe': sharpe}

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
bounds = [(0.0, 1.0)] * len(mean_vec)

result_max_sharpe = minimize(
    lambda w: -portfolio_stats(w)['sharpe'],
    weights0,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
)

result_min_var = minimize(
    lambda w: portfolio_stats(w)['variance'],
    weights0,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
)

if not result_max_sharpe.success:
    raise RuntimeError(result_max_sharpe.message)
if not result_min_var.success:
    raise RuntimeError(result_min_var.message)

optimal_weights = pd.DataFrame({
    'ticker': returns.columns,
    'weight': result_max_sharpe.x,
}).sort_values('weight', ascending=False)
optimal_weights['weight_pct'] = optimal_weights['weight'] * 100

min_var_weights = pd.DataFrame({
    'ticker': returns.columns,
    'weight': result_min_var.x,
}).sort_values('weight', ascending=False)
min_var_weights['weight_pct'] = min_var_weights['weight'] * 100

print('Max-Sharpe optimized weights:')
print(optimal_weights.round(4))
print('\nMinimum-variance weights:')
print(min_var_weights.round(4))


# 5) Plot the efficient frontier and summarize the final allocation

This final section visualizes the efficient frontier and translates the optimized weights into an intuitive summary of the strategic asset allocation.


In [ ]:
frontier = []
for target_return in np.linspace(annualized_return.min(), annualized_return.max(), 30):
    target_constraint = {'type': 'eq', 'fun': lambda w, tr=target_return: w @ annualized_return - tr}
    res = minimize(
        lambda w: portfolio_stats(w)['variance'],
        weights0,
        method='SLSQP',
        bounds=bounds,
        constraints=[*constraints, target_constraint],
    )
    if res.success:
        stats = portfolio_stats(res.x)
        frontier.append({
            'target_return': target_return,
            'volatility': stats['volatility'],
            'sharpe': stats['sharpe'],
        })

frontier_df = pd.DataFrame(frontier)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(frontier_df['volatility'], frontier_df['target_return'], 'o-', linewidth=2)
ax.scatter(
    portfolio_stats(result_max_sharpe.x)['volatility'],
    portfolio_stats(result_max_sharpe.x)['return'],
    color='red', s=100, label='Max Sharpe',
)
ax.set_title('Efficient Frontier for the Strategic Asset Allocation')
ax.set_xlabel('Annualized Volatility')
ax.set_ylabel('Annualized Expected Return')
ax.legend()
plt.show()

max_sharpe_stats = portfolio_stats(result_max_sharpe.x)
print('Max-Sharpe portfolio summary:')
print({
    'expected_return': round(max_sharpe_stats['return'], 4),
    'volatility': round(max_sharpe_stats['volatility'], 4),
    'sharpe': round(max_sharpe_stats['sharpe'], 4),
})

optimal_weights_by_asset_class = (
    optimal_weights.set_index('ticker')
    .join(portfolio.as_dataframe().set_index('ticker')[['asset_class']], how='left')
    .groupby('asset_class')['weight']
    .sum()
    .sort_values(ascending=False)
)

print('\nAllocation by asset class:')
print(optimal_weights_by_asset_class.round(4))
